# Phase 1 — Session 1 : LGN, TCL, Maximum de Vraisemblance

**Objectif** : poser les fondations probabilistes du ML.  
**Niveau** : MP* / M1 maths appliquées  
**Deadline** : 17 avril 2026

---

## Ce qu'il faut retenir

| Concept | En une phrase |
|---|---|
| **LGN** | La moyenne empirique converge vers la vraie moyenne. Vitesse : $1/\sqrt{n}$ |
| **TCL** | Quelle que soit la distribution, $\bar{X}_n$ standardisée converge vers $\mathcal{N}(0,1)$ |
| **MLE** | Choisir $\theta$ qui maximise $\sum \log p(x_i|\theta)$ |
| **Cross-entropy** | $H(\hat{p}, p) = -\sum \hat{p}(x)\log p(x)$ — minimiser la cross-entropy = maximiser la log-vraisemblance |
| **Lien ML** | `F.cross_entropy` de PyTorch = negative log-likelihood d'une Bernoulli |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

%matplotlib inline
np.random.seed(42)

---

## 1. Loi des Grands Nombres (LGN)

Soit $(X_1, ..., X_n)$ i.i.d. d'espérance $\mu$ et variance $\sigma^2 < \infty$.

**LGN faible** (convergence en probabilité) :
$$\forall \varepsilon > 0, \quad \mathbb{P}(|\bar{X}_n - \mu| > \varepsilon) \xrightarrow[n \to \infty]{} 0$$

**Ce que ça signifie en ML** : la loss calculée sur un batch converge vers la vraie loss sur la distribution entière. C'est la justification théorique de l'entraînement par batch.

**Vitesse de convergence** : $\text{Var}(\bar{X}_n) = \sigma^2/n$ → l'écart-type décroît en $1/\sqrt{n}$.  
Doubler la précision coûte **4x** plus de données.

In [ ]:
mu_vraie = 3.5
ns = np.arange(1, 5001)
moyennes = np.cumsum(np.random.exponential(mu_vraie, 5000)) / ns

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ns, moyennes, color='steelblue', linewidth=1, label='Moyenne empirique')
ax.axhline(mu_vraie, color='red', linestyle='--', label=f'Vraie moyenne μ={mu_vraie}')
ax.set_xlabel("Nombre d'observations n")
ax.set_ylabel("Moyenne empirique")
ax.set_title("Convergence de la moyenne empirique — Loi des Grands Nombres")
ax.legend()
plt.tight_layout()

---

## 2. Théorème Central Limite (TCL)

$$\sqrt{n}\,\frac{\bar{X}_n - \mu}{\sigma} \xrightarrow[n \to \infty]{\mathcal{L}} \mathcal{N}(0, 1)$$

**Ce que ça signifie en ML** :
- Les intervalles de confiance reposent sur le TCL
- La gaussienne est l'attracteur universel des sommes de v.a. — c'est pourquoi elle apparaît partout
- On choisit une loi exponentielle (très asymétrique) pour montrer que la convergence est universelle

In [ ]:
n_simulations = 10000
ns_tcl = [1, 10, 100, 1000]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, n in zip(axes, ns_tcl):
    moyennes = np.mean(np.random.exponential(1, size=(n_simulations, n)), axis=1)
    Z = (moyennes - moyennes.mean()) / moyennes.std()
    ax.hist(Z, bins=50, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    x = np.linspace(-4, 4, 200)
    ax.plot(x, np.exp(-x**2/2) / np.sqrt(2*np.pi), color='red', linewidth=2)
    ax.set_title(f'n = {n}')
    ax.set_xlim(-4, 4)

plt.suptitle("Convergence vers la gaussienne — TCL sur loi exponentielle")
plt.tight_layout()

---

## 3. Maximum de Vraisemblance (MLE)

**Le problème** : on observe $x_1,...,x_n$ supposés i.i.d. de loi $p(x|\theta)$. Estimer $\theta$.

**L'estimateur MLE** :
$$\hat{\theta}_{MLE} = \arg\max_\theta \sum_{i=1}^n \log p(x_i \mid \theta)$$

### MLE d'une gaussienne

**Log-vraisemblance** :
$$\ell(\mu, \sigma^2) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (x_i - \mu)^2$$

**Dérivation** :
$$\frac{\partial \ell}{\partial \mu} = 0 \implies \hat{\mu}_{MLE} = \bar{x}$$
$$\frac{\partial \ell}{\partial \sigma^2} = 0 \implies \hat{\sigma}^2_{MLE} = \frac{1}{n}\sum(x_i - \bar{x})^2$$

**Note** : le MLE divise par $n$ (pas $n-1$) → estimateur biaisé de $\sigma^2$ (session 2).

In [ ]:
mu_vraie, sigma_vraie = 5.0, 2.0
n = 1000
X = np.random.normal(mu_vraie, sigma_vraie, n)

mu_mle    = X.mean()
sigma_mle = np.sqrt(((X - mu_mle)**2).mean())

print(f"Vraie μ  : {mu_vraie:.4f}  |  MLE μ̂  : {mu_mle:.4f}")
print(f"Vraie σ  : {sigma_vraie:.4f}  |  MLE σ̂  : {sigma_mle:.4f}")

mus = np.linspace(4, 6, 200)
log_vrai = [-0.5 * np.sum((X - mu)**2) / sigma_vraie**2 for mu in mus]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mus, log_vrai, color='steelblue', linewidth=2)
ax.axvline(mu_mle, color='red', linestyle='--', label=f'MLE μ̂ = {mu_mle:.3f}')
ax.axvline(mu_vraie, color='green', linestyle=':', label=f'Vraie μ = {mu_vraie}')
ax.set_xlabel("μ")
ax.set_ylabel("Log-vraisemblance")
ax.set_title("Log-vraisemblance en fonction de μ — Gaussienne")
ax.legend()
plt.tight_layout()

---

## Exercice 1 — LGN & TCL sur une Bernoulli

### Q1 — Preuve : variance de $\bar{X}_n$ pour $X_i \sim \text{Bernoulli}(p)$

Par i.i.d. et propriétés de la variance :

$$\text{Var}(\bar{X}_n) = \text{Var}\left(\frac{1}{n}\sum_{i=1}^n X_i\right) = \frac{1}{n^2}\sum_{i=1}^n \text{Var}(X_i) = \frac{p(1-p)}{n}$$

**Implication** : l'écart-type de $\bar{X}_n$ est $\sqrt{p(1-p)/n}$ — décroît en $1/\sqrt{n}$.  
Pour $p=0.3$ : diviser l'erreur par 2 nécessite 4x plus de données.

In [ ]:
# Q2 — Vérification empirique du TCL sur Bernoulli(0.3)
n_simulations = 10000
ns_bern = [1, 10, 100, 1000]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for ax, n in zip(axes, ns_bern):
    moyennes = np.mean(np.random.binomial(1, 0.3, size=(n_simulations, n)), axis=1)
    Z = (moyennes - moyennes.mean()) / moyennes.std()
    ax.hist(Z, bins=50, density=True, color='steelblue', alpha=0.7, edgecolor='white')
    x = np.linspace(-4, 4, 200)
    ax.plot(x, np.exp(-x**2/2) / np.sqrt(2*np.pi), color='red', linewidth=2)
    ax.set_title(f'n = {n}')
    ax.set_xlim(-4, 4)

plt.suptitle("Convergence vers la gaussienne — TCL sur Bernoulli(0.3)")
plt.tight_layout()

# Observation : à partir de n=100, l'approximation gaussienne est très bonne.
# Règle empirique pour Bernoulli : np > 5 et n(1-p) > 5 → n > 17 pour p=0.3

---

## Exercice 2 — MLE de la loi de Poisson

### Q1 — Preuve : $\hat{\lambda}_{MLE}$

Pour $X_i \sim \mathcal{P}(\lambda)$, $P(X=k) = e^{-\lambda}\lambda^k/k!$

**Log-vraisemblance** :
$$\ell(\lambda) = \sum_{i=1}^n \left[-\lambda + x_i\log\lambda - \log(x_i!)\right] = -n\lambda + \left(\sum x_i\right)\log\lambda - \sum\log(x_i!)$$

**Dérivation** :
$$\frac{d\ell}{d\lambda} = -n + \frac{\sum x_i}{\lambda} = 0 \implies \hat{\lambda}_{MLE} = \frac{1}{n}\sum x_i = \bar{x}$$

La moyenne empirique est le MLE de $\lambda$ — cohérent car $\mathbb{E}[X] = \lambda$ pour une Poisson.

In [ ]:
# Q2 — Vérification numérique
n = 500
L_vraie = 4.2
X = np.random.poisson(L_vraie, n)
X_factorial = np.vectorize(math.factorial)(X)

L_mle = X.mean()
print(f"Vraie λ  : {L_vraie:.4f}  |  MLE λ̂  : {L_mle:.4f}")

lams = np.linspace(2, 8, 200)
log_vraisemblances = [
    -n * lam + np.sum(X) * np.log(lam) - np.sum(np.log(X_factorial))
    for lam in lams
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(lams, log_vraisemblances, color='steelblue', linewidth=2)
ax.axvline(L_mle, color='red', linestyle='--', label=f'MLE λ̂ = {L_mle:.3f}')
ax.axvline(L_vraie, color='green', linestyle=':', label=f'Vraie λ = {L_vraie}')
ax.set_xlabel("λ")
ax.set_ylabel("Log-vraisemblance")
ax.set_title("Log-vraisemblance en fonction de λ — Poisson")
ax.legend()
plt.tight_layout()

---

## Exercice 3 — MLE Bernoulli & Cross-entropy

### Q1 — Preuve : $\hat{p}_{MLE} = \bar{x}$ et lien avec la cross-entropy

**MLE de $p$** pour $X_i \sim \text{Bernoulli}(p)$ :

$$\ell(p) = \sum x_i \log p + (n - \sum x_i)\log(1-p)$$

$$\frac{d\ell}{dp} = \frac{\sum x_i}{p} - \frac{n - \sum x_i}{1-p} = 0 \implies \hat{p}_{MLE} = \bar{x}$$

**Lien avec la cross-entropy** :

La cross-entropy entre la distribution empirique $\hat{p}$ et le modèle $p$ :
$$H(\hat{p}, p) = -\bar{x}\log p - (1-\bar{x})\log(1-p)$$

Et $\ell(p) = n\left[\bar{x}\log p + (1-\bar{x})\log(1-p)\right] = -n \cdot H(\hat{p}, p)$

**Conclusion** : maximiser $\ell(p)$ ↔ minimiser $H(\hat{p}, p)$.  
`F.cross_entropy` de PyTorch = $-\ell(w)/n$ — la negative log-likelihood.

In [ ]:
# Q2 — Vérification numérique
n = 200
p_vraie = 0.7
X = np.random.binomial(1, p_vraie, n)

p_mle = X.mean()
print(f"Vraie p  : {p_vraie:.4f}  |  MLE p̂  : {p_mle:.4f}")

ps = np.linspace(0.01, 0.99, 200)
log_vraisemblances = [
    np.log(p) * np.sum(X) + np.log(1-p) * (n - np.sum(X))
    for p in ps
]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ps, log_vraisemblances, color='steelblue', linewidth=2)
ax.axvline(p_mle, color='red', linestyle='--', label=f'MLE p̂ = {p_mle:.3f}')
ax.axvline(p_vraie, color='green', linestyle=':', label=f'Vraie p = {p_vraie}')
ax.set_xlabel("p")
ax.set_ylabel("Log-vraisemblance")
ax.set_title("Log-vraisemblance en fonction de p — Bernoulli")
ax.legend()
plt.tight_layout()

---

## Ce qu'il faut retenir — Synthèse

### Les 3 résultats fondamentaux

**1. LGN** — $\bar{X}_n \to \mu$ en probabilité. Vitesse : $\text{std}(\bar{X}_n) = \sigma/\sqrt{n}$.

**2. TCL** — $\sqrt{n}(\bar{X}_n - \mu)/\sigma \to \mathcal{N}(0,1)$. Universel — peu importe la distribution de départ.

**3. MLE** — $\hat{\theta} = \arg\max \sum \log p(x_i|\theta)$. Pour les trois lois vues :

| Loi | MLE |
|---|---|
| $\mathcal{N}(\mu, \sigma^2)$ | $\hat{\mu} = \bar{x}$, $\hat{\sigma}^2 = \frac{1}{n}\sum(x_i-\bar{x})^2$ |
| $\mathcal{P}(\lambda)$ | $\hat{\lambda} = \bar{x}$ |
| $\text{Bernoulli}(p)$ | $\hat{p} = \bar{x}$ |

### Le lien avec le ML

- **Loss = negative log-likelihood**. `F.cross_entropy` = $-\ell(w)/n$.
- **Entraîner un modèle = faire du MLE** sur les paramètres $w$.
- **La LGN justifie** que minimiser la loss sur un batch ≈ minimiser la vraie loss.
- **Le TCL justifie** les intervalles de confiance et les tests statistiques (session 3).

### Ce qui arrive en session 2

Le MLE de $\sigma^2$ divise par $n$ — mais l'estimateur sans biais divise par $n-1$. Pourquoi ?  
C'est la question centrale de la session 2 : **biais, variance et efficacité des estimateurs**.